# Notebook 02: Huis Data Exploratie 

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "locatie_data"

In [ ]:
df_huis = pd.read_parquet(
    DATA_DIR / "huis788.parquet"
) 

### Algemene controle

In [3]:
print("Shape:", df_huis.shape)
print("Chronologisch:", df_huis["timestamp"].is_monotonic_increasing)

df_huis.info()

Shape: (3057890, 24)
Chronologisch: False
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3057890 entries, 0 to 3057889
Data columns (total 24 columns):
 #   Column        Dtype                           
---  ------        -----                           
 0   timestamp     datetime64[ns, Europe/Amsterdam]
 1   CO2           float64                         
 2   CTYPE         object                          
 3   ENAME         object                          
 4   ENE           float64                         
 5   EONNAME       object                          
 6   ETYPE         object                          
 7   GNAME         object                          
 8   POW           float64                         
 9   CO2_IMP       float64                         
 10  COST_IMP      float64                         
 11  CTYPEC        object                          
 12  ENE_CNT_IMP   float64                         
 13  ENE_IMP       float64                         
 14  CO2_EXP 

In [4]:
df_huis.head()

,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,CO2_IMP,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,boiler-16eda211b788,NaN,eiot-16eda211b788,boiler,SERENE,6.504839,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,2024-03-08 01:05:00+01:00,0.099741,electricity,boiler-16eda211b788,0.542070,eiot-16eda211b788,boiler,SERENE,6.490000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,2024-03-08 01:10:00+01:00,0.099513,electricity,boiler-16eda211b788,0.540833,eiot-16eda211b788,boiler,SERENE,6.461972,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
3,2024-03-08 01:15:00+01:00,0.099084,electricity,boiler-16eda211b788,0.538498,eiot-16eda211b788,boiler,SERENE,6.477778,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
4,2024-03-08 01:20:00+01:00,0.099326,electricity,boiler-16eda211b788,0.539815,eiot-16eda211b788,boiler,SERENE,6.461039,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN


### Overzicht van de meetreeksen

In [5]:
# maakt per ENAME een samenvatting vd timestamps 
overzicht = (df_huis.groupby("ENAME").agg(
        aantal_rijen=("timestamp", "size"), # telt hoeveel rijen er per ENAME zijn 
        unieke_timestamps=("timestamp", "nunique"), # telt hoeveel verschillende timestamps er per ENAME zijn 
        eerste_meting=("timestamp", "min"), # eerste timestamp die voorkomt per ENAME 
        laatste_meting=("timestamp", "max") # laatste timestamp die voorkomt per ENAME
    )
)

# berekent hoeveel extra rijen er zijn doordat timestamps vaker dan 1 keer voorkomen 
overzicht["dubbele_timestamps"] = (overzicht["aantal_rijen"] - overzicht["unieke_timestamps"])

overzicht.sort_values("aantal_rijen", ascending=False)

,aantal_rijen,unieke_timestamps,eerste_meting,laatste_meting,dubbele_timestamps
ENAME,,,,,
overig-16eda211b788,398790,230149,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,168641
totaalverbruik-16eda211b788,397154,228499,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,168655
wasmachine-16eda211b788,381248,214270,2024-03-08 01:05:00+01:00,2026-09-10 01:55:00+02:00,166978
omvormer-16eda211b788,368720,200703,2024-03-08 06:55:00+01:00,2026-09-10 01:55:00+02:00,168017
boiler-16eda211b788,362465,195413,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,167052
wcdskeukeneilandenvaatwasser-16eda211b788,361006,194555,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,166451
hoofdmeter-16eda211b788,318076,188249,2024-03-08 01:00:00+01:00,2026-09-07 08:05:00+02:00,129827
priceele_c911efafaa,171217,171217,2025-01-14 09:20:00+01:00,2026-09-10 01:55:00+02:00,0
co2ele_c911efafaa,167079,167070,2024-06-20 12:15:00+02:00,2026-09-10 01:55:00+02:00,9


### Totaalverbruik bekijken

In [6]:
TOTAALVERBRUIK = "totaalverbruik-16eda211b788"

# aparte dataframe met alleen de rijen van totaalverbruik
df_totaal = (df_huis[df_huis["ENAME"] == TOTAALVERBRUIK].copy()
             .sort_values("timestamp").reset_index(drop=True)
)

print("Aantal rijen:", len(df_totaal))
print("Unieke timestamps:", df_totaal["timestamp"].nunique())

# berekent hoeveel extra rijen er zijn doordat timestamps vaker voorkomen
print("Dubbele timestamps:",
    len(df_totaal) - df_totaal["timestamp"].nunique()
)

df_totaal.head()

Aantal rijen: 397154
Unieke timestamps: 228499
Dubbele timestamps: 168655


,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,CO2_IMP,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,100.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,2024-03-08 01:05:00+01:00,1.533333,electricity,totaalverbruik-16eda211b788,8.333333,eiot-16eda211b788,powermeter,SERENE,40.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,2024-03-08 01:10:00+01:00,0.613333,electricity,totaalverbruik-16eda211b788,3.333333,eiot-16eda211b788,powermeter,SERENE,45.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
3,2024-03-08 01:15:00+01:00,0.690000,electricity,totaalverbruik-16eda211b788,3.750000,eiot-16eda211b788,powermeter,SERENE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
4,2024-03-08 01:20:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,45.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN


In [7]:
df_totaal["timestamp"].value_counts().value_counts().sort_index()

count
1     59844
2    168655
Name: count, dtype: int64

In [8]:
print(df_huis.columns.tolist())

['timestamp', 'CO2', 'CTYPE', 'ENAME', 'ENE', 'EONNAME', 'ETYPE', 'GNAME', 'POW', 'CO2_IMP', 'COST_IMP', 'CTYPEC', 'ENE_CNT_IMP', 'ENE_IMP', 'CO2_EXP', 'COST', 'COST_EXP', 'ENE_CNT_EXP', 'ENE_EXP', 'CO2EQ', 'ESTYPE', 'forecast.ENE', 'forecast.POW', 'PRICE']


In [9]:
print(df_totaal.columns.tolist())
df_totaal.head()

['timestamp', 'CO2', 'CTYPE', 'ENAME', 'ENE', 'EONNAME', 'ETYPE', 'GNAME', 'POW', 'CO2_IMP', 'COST_IMP', 'CTYPEC', 'ENE_CNT_IMP', 'ENE_IMP', 'CO2_EXP', 'COST', 'COST_EXP', 'ENE_CNT_EXP', 'ENE_EXP', 'CO2EQ', 'ESTYPE', 'forecast.ENE', 'forecast.POW', 'PRICE']


,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,CO2_IMP,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,100.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,2024-03-08 01:05:00+01:00,1.533333,electricity,totaalverbruik-16eda211b788,8.333333,eiot-16eda211b788,powermeter,SERENE,40.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,2024-03-08 01:10:00+01:00,0.613333,electricity,totaalverbruik-16eda211b788,3.333333,eiot-16eda211b788,powermeter,SERENE,45.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
3,2024-03-08 01:15:00+01:00,0.690000,electricity,totaalverbruik-16eda211b788,3.750000,eiot-16eda211b788,powermeter,SERENE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
4,2024-03-08 01:20:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,45.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
